In [11]:
# TODO
#   1. Figure out a RAG approach
#   2. Figure out guardrails.

## RAG approach
1. Dense/vector search for semantic similarity
2. BM25/keyword search for exact medical terms
3. Metadata filtering for trusted sources/topic/date
4. Reranking to choose the strongest evidence
5. Evidence-grounded generation where the LLM is explicitly restricted to retrieved material

Original query: “I saw on TikTok that taking 10 mg of melatonin every night fixes sleep deprivation. Is that true?”
       ↓
Claim extraction
       ↓
"10 mg melatonin every night fixes sleep deprivation"
       ↓
Query rewriting
       ↓
1. melatonin effectiveness for sleep deprivation
2. melatonin dosage young adults
3. melatonin long-term safety
       ↓
Hybrid retrieval
   ┌───────────────┐
   │ Dense search  │
   │ BM25 search   │
   └──────┬────────┘
          ↓
Trusted-source filter
          ↓
Reranker
          ↓
Top 3–5 evidence chunks
          ↓
Evidence sufficiency check
          ↓
LLM
          ↓
Grounded educational response + citations

## Overall structure for the project.

                    USER
                      │
                      ▼
             Text / Image Input
                      │
                      ▼
              INPUT GUARDRAIL
         privacy / safety / scope
                      │
                      ▼
              CLAIM EXTRACTION
                      │
                      ▼
           QUERY CLASSIFICATION
                     /              
                Supplements          
                    |       
                    │
                    ▼
              QUERY REWRITING
                    │
                    ▼
        ┌───────────────────────┐
        │      HYBRID RAG       │
        │                       │
        │ Dense vector search   │
        │          +            │
        │ BM25 keyword search   │
        └───────────┬───────────┘
                    │
                    ▼
           SOURCE/METADATA FILTER
                    │
                    ▼
                RERANKER
                    │
                    ▼
          EVIDENCE SUFFICIENT?
              /           \
            No             Yes
            │               │
      Rewrite/search        ▼
         again       EVIDENCE PACKAGE
                            │
                            ▼
                           LLM
                            │
                            ▼
                    OUTPUT GUARDRAIL
                   /        |        \
             grounded?   safe?    citations?
                            │
                            ▼
                  STRUCTURED RESPONSE
                  • claim assessment
                  • explanation
                  • evidence strength
                  • uncertainty
                  • sources
                            │
                            ▼
                           USER

## RAG technique:
Use Corrective RAG.

Retrieve evidence
       ↓
Evaluate retrieval quality
       ↓
Is the evidence good?
   /            \
 Yes            No
 │               │
Use it      Rewrite query
 │               │
 │          Retrieve again
 │               │
 └───────┬───────┘
         ▼
     Generate

## Imports + Model prepping

In [12]:
import os
from pathlib import Path
import pandas as pd
import chromadb
from openai import OpenAI
from sentence_transformers import SentenceTransformer

In [13]:
client = OpenAI(
    api_key=os.environ["COMP3110_API_KEY"],
    base_url=os.environ["COMP3110_MODEL_BASE_URL"]
)

MODEL = "scx-gpt-oss-120b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


c:\Users\alexa\OneDrive\Documents\medifo\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alexa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\alexa\OneDrive\Documents\medifo\.venv\Lib\site-packages\huggingface_hub\fi

## Grab Documents


In [14]:
# TODO grab at least 10 trusted documents about supplements
# Try to do each document in this structure: id, title, source, url, topic, text.

# documents = [
#     {
#         "id": "",
#         "title": "",
#         "source": "",
#         "url": "",
#         "topic": "",
#         "text": ""
#     },
# ]

documents = [
    {
        "id": "nih_melatonin",
        "title": "Melatonin: What You Need To Know",
        "source": "NIH",
        "url": "https://example.com/nih-melatonin",
        "topic": "sleep",
        "text": """
        Melatonin is a hormone produced by the brain in response
        to darkness. It helps with the timing of circadian rhythms
        and sleep. Research suggests that melatonin may be useful
        for certain sleep-related conditions...
        """
    },
    {
        "id": "sleep_deprivation_1",
        "title": "Sleep Deprivation and Deficiency",
        "source": "NIH",
        "url": "https://example.com/sleep-deprivation",
        "topic": "sleep",
        "text": """
        Sleep deprivation occurs when a person does not get enough
        sleep. Insufficient sleep may affect attention, memory,
        reaction time and other aspects of health...
        """
    },
    {
        "id": "magnesium_1",
        "title": "Magnesium Fact Sheet",
        "source": "NIH",
        "url": "https://example.com/magnesium",
        "topic": "supplements",
        "text": """
        Magnesium is an essential mineral involved in many processes
        in the body. Magnesium supplements are marketed for a range
        of uses, although evidence differs depending on the claimed
        benefit...
        """
    }
]

In [15]:
documents_df = pd.DataFrame(documents)
documents_df[["id", "title", "source", "topic"]]

,id,title,source,topic
0,nih_melatonin,Melatonin: What You Need To Know,NIH,sleep
1,sleep_deprivation_1,Sleep Deprivation and Deficiency,NIH,sleep
2,magnesium_1,Magnesium Fact Sheet,NIH,supplements


# Chunk Documents

### What overlap means
Overlapping is repeating a small portion (50 words) from the end of one chunk to the start of the next

### What does this do?
When a document is split in chunks, no sentence, idea or phrases are cut awkwardly when a document is divided. It also preserves context and allows the system to piece information together better.


In [16]:
def chunk_text(
    text, 
    chunk_size=300,
    overlap=50
):
    chunks = []
    words = text.split()
    start = 0

    while start < len(words):
        chunk = words[start:start + chunk_size]
        chunks.append(" ".join(chunk))
        start += chunk_size - overlap
    return chunks

In [17]:
chunks = []

for document in documents:

    document_chunks = chunk_text(
        document["text"]
    )

    for index, text in enumerate(
        document_chunks
    ):

        chunks.append(
            {
                "id": f"{document['id']}_{index}",
                "text": text,
                "title": document["title"],
                "source": document["source"],
                "url": document["url"],
                "topic": document["topic"],
            }
        )

In [18]:
chunks_df = pd.DataFrame(chunks)
chunks_df.head()

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Documents: 3
Chunks: 3


## Embeddings


In [19]:
def create_embedding(text: str) -> list[float]:
    embedding = embedding_model.encode(text)
    return embedding.tolist()

In [20]:
# Test embedding
test_embedding = create_embedding("Does magnesium improve sleep?")

print(len(test_embedding))
print(test_embedding[:5])

384
[-0.02096354030072689, 0.04133430868387222, 0.0036191947292536497, 0.11398790031671524, 0.043296318501234055]


## Vector Database

In [21]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="medical_evidence")

## Embed and store the chunks

In [22]:
for chunk in chunks:

    embedding = create_embedding(chunk["text"])

    collection.add(
        ids=[chunk["id"]],
        documents=[chunk["text"]],
        embeddings=[embedding],
        metadatas=[
            {
                "title": chunk["title"],
                "source": chunk["source"],
                "url": chunk["url"],
                "topic": chunk["topic"],
            }
        ],
    )

# Check collection
collection.count()

3

## Testing Vector Retrieval

In [23]:
def vector_search(query, top_k=5,):
    query_embedding = create_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    return results

# Testing
results = vector_search("Can magnesium help me sleep?")
results

{'ids': [['magnesium_1_0', 'nih_melatonin_0', 'sleep_deprivation_1_0']],
 'embeddings': None,
 'documents': [['Magnesium is an essential mineral involved in many processes in the body. Magnesium supplements are marketed for a range of uses, although evidence differs depending on the claimed benefit...',
   'Melatonin is a hormone produced by the brain in response to darkness. It helps with the timing of circadian rhythms and sleep. Research suggests that melatonin may be useful for certain sleep-related conditions...',
   'Sleep deprivation occurs when a person does not get enough sleep. Insufficient sleep may affect attention, memory, reaction time and other aspects of health...']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'title': 'Magnesium Fact Sheet',
    'url': 'https://example.com/magnesium',
    'topic': 'supplements',
    'source': 'NIH'},
   {'url': 'https://example.com/nih-melatonin',
    'topic': 'sleep',
    'titl

## Retrieval inspection


In [24]:
def show_results(results):

    rows = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for text, metadata, distance in zip(
        documents,
        metadatas,
        distances,
    ):
        rows.append(
            {
                "source": metadata["source"],
                "title": metadata["title"],
                "topic": metadata["topic"],
                "distance": distance,
                "text": text,
            }
        )

    return pd.DataFrame(rows)

show_results(results)

,source,title,topic,distance,text
0,NIH,Magnesium Fact Sheet,supplements,0.970100,Magnesium is an essential mineral involved in ...
1,NIH,Melatonin: What You Need To Know,sleep,1.012042,Melatonin is a hormone produced by the brain i...
2,NIH,Sleep Deprivation and Deficiency,sleep,1.197847,Sleep deprivation occurs when a person does no...


## Topic filtering

In [25]:
def vector_search(
    query,
    topic=None,
    top_k=5,
):

    query_embedding = create_embedding(query)

    query_args = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
    }

    if topic is not None:
        query_args["where"] = {
            "topic": topic
        }

    return collection.query(**query_args)

## Testing
results = vector_search(
    query="Does magnesium help sleep?",
    topic="supplements",
)

show_results(results)

,source,title,topic,distance,text
0,NIH,Magnesium Fact Sheet,supplements,0.943174,Magnesium is an essential mineral involved in ...


## Claim Extraction

In [26]:
def extract_claims(user_input):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
You extract medical claims from user messages.

Return each independent factual medical claim on a separate line.
Do not evaluate whether the claim is correct.
"""
            },
            {
                "role": "user",
                "content": user_input,
            },
        ],
        temperature=0,
    )

    output = (
        response
        .choices[0]
        .message
        .content
    )

    claims = [
        claim.strip("- ").strip()
        for claim in output.splitlines()
        if claim.strip()
    ]

    return claims

# Testing
user_input = """
    My friend said magnesium glycinate makes you fall asleep
    because it increases melatonin.
"""
claims = extract_claims(user_input)
claims

['Magnesium glycinate makes you fall asleep.',
 'Magnesium glycinate increases melatonin.']

## Query Rewriting

In [27]:
def rewrite_query(claim):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
Convert medical claims into concise evidence-search queries.

Return only the search query.
"""
            },
            {
                "role": "user",
                "content": claim,
            },
        ],
        temperature=0,
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

for claim in claims:
    query = rewrite_query(claim)
    print("Claim:", claim)
    print("Query:", query)
    print()

Claim: Magnesium glycinate makes you fall asleep.
Query: magnesium glycinate sleep onset insomnia evidence

Claim: Magnesium glycinate increases melatonin.
Query: magnesium glycinate effect on melatonin levels



## Evidence retrieval

In [28]:
all_evidence = []

for claim in claims:

    query = rewrite_query(claim)
    results = vector_search(
        query=query,
        top_k=5,
    )
    evidence_df = show_results(results)

    evidence_df["claim"] = claim
    evidence_df["query"] = query

    all_evidence.append(evidence_df)

In [29]:
# Combine
evidence_df = pd.concat(
    all_evidence,
    ignore_index=True,
)
evidence_df

,source,title,topic,distance,text,claim,query
0,NIH,Magnesium Fact Sheet,supplements,1.100213,Magnesium is an essential mineral involved in ...,Magnesium glycinate makes you fall asleep.,magnesium glycinate sleep onset insomnia evidence
1,NIH,Sleep Deprivation and Deficiency,sleep,1.146034,Sleep deprivation occurs when a person does no...,Magnesium glycinate makes you fall asleep.,magnesium glycinate sleep onset insomnia evidence
2,NIH,Melatonin: What You Need To Know,sleep,1.168827,Melatonin is a hormone produced by the brain i...,Magnesium glycinate makes you fall asleep.,magnesium glycinate sleep onset insomnia evidence
3,NIH,Melatonin: What You Need To Know,sleep,0.835920,Melatonin is a hormone produced by the brain i...,Magnesium glycinate increases melatonin.,magnesium glycinate effect on melatonin levels
4,NIH,Magnesium Fact Sheet,supplements,1.096392,Magnesium is an essential mineral involved in ...,Magnesium glycinate increases melatonin.,magnesium glycinate effect on melatonin levels
5,NIH,Sleep Deprivation and Deficiency,sleep,1.452933,Sleep deprivation occurs when a person does no...,Magnesium glycinate increases melatonin.,magnesium glycinate effect on melatonin levels


## Context Formatting

In [30]:
def format_context(results):

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    sections = []

    for i, (text, metadata) in enumerate(
        zip(documents, metadatas),
        start=1,
    ):

        section = f"""
SOURCE {i}

Title:
{metadata["title"]}

Organisation:
{metadata["source"]}

URL:
{metadata["url"]}

Evidence:
{text}
"""

        sections.append(section)

    return "\n".join(
        sections
    )

In [31]:
# Testing
results = vector_search(
    "magnesium supplementation sleep quality"
)
context = format_context(results)
print(context)


SOURCE 1

Title:
Magnesium Fact Sheet

Organisation:
NIH

URL:
https://example.com/magnesium

Evidence:
Magnesium is an essential mineral involved in many processes in the body. Magnesium supplements are marketed for a range of uses, although evidence differs depending on the claimed benefit...


SOURCE 2

Title:
Melatonin: What You Need To Know

Organisation:
NIH

URL:
https://example.com/nih-melatonin

Evidence:
Melatonin is a hormone produced by the brain in response to darkness. It helps with the timing of circadian rhythms and sleep. Research suggests that melatonin may be useful for certain sleep-related conditions...


SOURCE 3

Title:
Sleep Deprivation and Deficiency

Organisation:
NIH

URL:
https://example.com/sleep-deprivation

Evidence:
Sleep deprivation occurs when a person does not get enough sleep. Insufficient sleep may affect attention, memory, reaction time and other aspects of health...



## RAG Generation

In [32]:
def generate_answer(
    question,
    context,
):

    system_prompt = """
You are an educational medical misinformation assistant aimed
primarily at young adults.

Your purpose is to explain medical evidence, not provide personalised
medical advice.

Rules:
- Use the retrieved evidence to support factual medical claims.
- Do not diagnose the user.
- Do not prescribe treatment.
- Clearly communicate uncertainty.
- Do not invent evidence.
- If the retrieved evidence is insufficient, say so.
- Cite evidence using [Source 1], [Source 2], etc.
- Explain unfamiliar medical terminology using simple language.

Use this format:

Claim assessment:
Short answer:
What the evidence says:
Important context:
Evidence confidence:
"""

    user_prompt = f"""
USER QUESTION

{question}


RETRIEVED EVIDENCE

{context}
"""

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],

        temperature=0.1,
    )

    return (
        response
        .choices[0]
        .message
        .content
    )

In [33]:
# Testing
answer = generate_answer(
    question=(
        "Does magnesium really help you sleep?"
    ),
    context=context,
)

print(answer)

**Claim assessment:**  
The currently retrieved evidence does not provide clear, direct proof that magnesium supplementation reliably improves sleep.

**Short answer:**  
There is limited and inconclusive research on magnesium for sleep; some studies suggest a possible benefit, but the overall evidence is weak and not definitive.

**What the evidence says:**  
- The NIH “Magnesium Fact Sheet” notes that magnesium supplements are marketed for many health purposes, but the strength of the scientific evidence varies by claim. It does **not** specifically confirm that magnesium improves sleep quality or duration【Source 1】.  
- The other two NIH sources discuss melatonin’s role in sleep timing and the effects of sleep deprivation, but they do not address magnesium at all【Source 2】【Source 3】.

**Important context:**  
- Magnesium is essential for many bodily functions, including nerve and muscle activity, and low magnesium levels can cause muscle cramps or restless legs, which might indirect

## RAG function

In [34]:
def rag_answer(
    user_input,
    top_k=5,
):

    results = vector_search(
        query=user_input,
        top_k=top_k,
    )

    context = format_context(results)

    answer = generate_answer(
        question=user_input,
        context=context,
    )

    return {
        "answer": answer,
        "results": results,
    }

In [35]:
# Testing
result = rag_answer("Does melatonin fix sleep deprivation?")
print(result["answer"])

**Claim assessment:**  
Melatonin can aid the timing of sleep but it does **not** “fix” sleep deprivation on its own.

**Short answer:**  
Melatonin may help you fall asleep more easily, especially when your sleep schedule is misaligned (e.g., jet lag or shift work), but it is not a cure for chronic sleep deprivation.

**What the evidence says:**  
- Melatonin is a hormone the brain releases in darkness that signals the body it’s time to sleep. Research shows it can be useful for certain sleep‑related conditions, such as adjusting circadian rhythms, jet lag, and some cases of delayed sleep phase syndrome【Source 1】.  
- The evidence does **not** indicate that taking melatonin restores the total amount of sleep lost from chronic sleep deprivation. Sleep deprivation itself is defined as getting insufficient sleep, which impairs attention, memory, reaction time, and overall health【Source 2】. No studies in the provided sources demonstrate that melatonin can fully compensate for these defici

## Claim-aware pipeline

In [36]:
def claim_rag_answer(
    user_input,
    top_k=5,
):

    claims = extract_claims(user_input)

    retrieved_results = []

    for claim in claims:

        query = rewrite_query(claim)

        results = vector_search(
            query=query,
            top_k=top_k,
        )

        retrieved_results.append(
            {
                "claim": claim,
                "query": query,
                "results": results,
            }
        )

    return retrieved_results

In [37]:
# Testing
result = claim_rag_answer(
    """
    Taking melatonin means you can sleep four hours
    a night without suffering from sleep deprivation.
    """
)

result

[{'claim': 'Taking melatonin means you can sleep four hours a night without suffering from sleep deprivation.',
  'query': 'melatonin four hours sleep deprivation evidence',
  'results': {'ids': [['nih_melatonin_0',
     'sleep_deprivation_1_0',
     'magnesium_1_0']],
   'embeddings': None,
   'documents': [['Melatonin is a hormone produced by the brain in response to darkness. It helps with the timing of circadian rhythms and sleep. Research suggests that melatonin may be useful for certain sleep-related conditions...',
     'Sleep deprivation occurs when a person does not get enough sleep. Insufficient sleep may affect attention, memory, reaction time and other aspects of health...',
     'Magnesium is an essential mineral involved in many processes in the body. Magnesium supplements are marketed for a range of uses, although evidence differs depending on the claimed benefit...']],
   'uris': None,
   'included': ['metadatas', 'documents', 'distances'],
   'data': None,
   'metadata

## Experimentation / Evaluation

In [38]:
test_questions = [
    "Does magnesium improve sleep?",
    "Can melatonin replace lost sleep?",
    "Can supplements prevent the effects of sleep deprivation?",
    "Is taking more melatonin always better?",
    "Can four hours of sleep be enough if I take supplements?",
]

In [39]:
for question in test_questions:

    print("=" * 80)
    print("QUESTION:")
    print(question)

    result = rag_answer(question)

    print("\nANSWER:")
    print(result["answer"])
    print()

QUESTION:
Does magnesium improve sleep?

ANSWER:
**Claim assessment:**  
Current publicly available evidence does not provide a clear, consistent answer to whether magnesium supplementation improves sleep.

**Short answer:**  
There is not enough high‑quality evidence to say definitively that magnesium improves sleep for most people.

**What the evidence says:**  
- The NIH “Magnesium Fact Sheet” notes that magnesium supplements are marketed for many health purposes, but the strength of the scientific evidence varies by claim [Source 2]. It does not give a definitive conclusion that magnesium reliably enhances sleep quality or duration.  
- The other retrieved sources discuss melatonin’s role in sleep regulation [Source 1] and the consequences of sleep deprivation [Source 3]; they do not address magnesium’s effect on sleep.

**Important context:**  
- Some individual studies (not included in the retrieved set) have suggested modest benefits of magnesium for people with specific conditi